# endgame-probe -- how does eat-rest-v1 die?

`tune_v1` found v1's numeric settings flat (80 variants, none better on fresh seeds), so the next gain has to come from a
behaviour change. This runs `external/candidates/endgame_probe.py`: the untouched baseline on fresh seeds (11000+), recording
a 50 s timeline of the world and the colony plus every death and birth, and prints what changes in the run-up to extinction.

16 games, roughly 5-8 minutes; runs in the foreground so the report lands in the cell. Resumable: rerunning skips finished
games, and raising `SEEDS` only plays the new ones. Mechanism study: nothing is written to `results/`.

**Cluster setup:** same as `parameter-tuning.ipynb` (`.env` with `GITHUB_TOKEN=<token>`).

In [1]:
import os

CLONE_DIR = "/home/jovyan/Nordic-AI-cup-2026"
if os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print(f"{CLONE_DIR} already cloned - skipping (use `git pull` there to update)")
else:
    # GitHub token is read from a git-ignored .env (GITHUB_TOKEN=...) in the kernel's cwd, or from the environment
    if os.path.isfile(".env"):
        for line in open(".env"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
    TOKEN = os.environ.get("GITHUB_TOKEN")
    if not TOKEN:
        raise RuntimeError(f"GITHUB_TOKEN not set - create {os.path.abspath('.env')} containing GITHUB_TOKEN=<token>")
    !git clone https://{TOKEN}@github.com/sjoeen/Nordic-AI-cup-2026.git {CLONE_DIR}

/home/jovyan/Nordic-AI-cup-2026 already cloned - skipping (use `git pull` there to update)


In [2]:
import glob
import os
import subprocess
import sys

os.chdir(CLONE_DIR)
!git fetch origin challenge-1V2
!git checkout challenge-1V2
!git pull origin challenge-1V2

# Must run from survival-simulator/ so `src`, `agents`, `training` import.
if os.path.basename(os.getcwd()) != "survival-simulator":
    candidates = sorted({os.path.realpath(p) for p in glob.glob(os.path.join(os.getcwd(), "**", "survival-simulator"), recursive=True)
                         if os.path.isfile(os.path.join(p, "requirements.txt"))})
    if len(candidates) != 1:
        raise RuntimeError(f"cwd is {os.getcwd()}; found {len(candidates)} survival-simulator checkouts {candidates} - %cd into the right one")
    os.chdir(candidates[0])

print("cwd:", os.getcwd())
sys.path.insert(0, os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 16 (delta 9), reused 13 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (16/16), 15.42 KiB | 1.54 MiB/s, done.
From https://github.com/sjoeen/Nordic-AI-cup-2026
 * branch            challenge-1V2 -> FETCH_HEAD
   ec433c1..a1e6b54  challenge-1V2 -> origin/challenge-1V2
M	Nordic-AI-Cup-2026-main/survival-simulator/results/index.csv
Already on 'challenge-1V2'
Your branch is behind 'origin/challenge-1V2' by 2 commits, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/sjoeen/Nordic-AI-cup-2026
 * branch            challenge-1V2 -> FETCH_HEAD
Updating ec433c1..a1e6b54
Fast-forward
 .../survival-simulator/endgame-probe.ipynb         | 165 +++++++++++
 .../external/candidates/endgame_probe.py           | 130 +++++++++
 .../external/candidates/tune_v2.py                 | 152 +++++++++++
 ...

In [3]:
!{sys.executable} -m pip install -r requirements.txt -r requirements-dev.txt

## Run the probe and print the report

In [ ]:
SEEDS = 16
!{sys.executable} -u external/candidates/endgame_probe.py --out logs/eg2/base --seeds {SEEDS} 2>&1 | grep -v "pkg_resources\|pygame"

## Endgame mode trial: `eat-rest-endgame` against the baseline on the same seeds

`external/candidates/eat-rest-endgame` is v1 plus an endgame mode (see its docstring). It reads the colony's own state:
ON when the smoothed mean energy drops under 175 or 3 or fewer agents are left (never before t=400), OFF again once the
colony has held mean energy above 205 with 5+ agents for 30 s. While ON it vetoes births that would leave the parent under
50 energy or happen with a predator within 130, and agents of 95 s or older are ordered to breed before old age takes their
energy. Run the baseline cell above first; `--compare` then prints the paired per-seed difference, when the mode first
switched on (`eg_first_on`), how often it switched on/off, and how often each rule fired.


In [ ]:
!{sys.executable} -u external/candidates/endgame_probe.py --out logs/eg2/eg --candidate eat-rest-endgame --seeds {SEEDS} --compare logs/eg2/base 2>&1 | grep -v "pkg_resources\|pygame"

## Variants of the endgame mode (each in its own folder, same seeds, compared with the baseline)

`--set` overrides the candidate's settings. `eg_overrides` holds baseline settings that apply only while the endgame is on.

In [ ]:
VARIANTS = {
    "eg_food":     '{"eg_birth_food": 150}',                        # also require a fruit within 150 to give birth
    "eg_nolegacy": '{"eg_legacy_age": 0}',                          # vetoes only, no ordered births by old agents
    "clock":       '{"eg_trigger": "clock"}',                       # trigger on expected predators (clock/100) vs living agents
    "sensed":      '{"eg_trigger": "sensed"}',                      # trigger on predators the agents sense (levels NOT calibrated yet)
    "eg_calm":     '{"eg_overrides": {"escape_radius": 80}}',       # flee later while the endgame is on
}
for name, overrides in VARIANTS.items():
    !{sys.executable} -u external/candidates/endgame_probe.py --out logs/eg2/{name} --candidate eat-rest-endgame --seeds {SEEDS} --set '{overrides}' --compare logs/eg2/base 2>&1 | grep -v "pkg_resources\|pygame" | grep -A12 "paired against"

## Report only (no new games)

In [ ]:
!{sys.executable} -u external/candidates/endgame_probe.py --out logs/eg2/base --seeds 0 2>&1 | grep -v "pkg_resources\|pygame"